Hacemos una recapitulacion del RAG que tenemos hasta ahora
- Descargamos el repo
- Parseamos los documentos
- Generamos los chunks
- Creamos el indice
- Lo relacionamos con los chunks

In [7]:
from openai import OpenAI

openai_client = OpenAI()

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


Creamos la funcion de busqueda

In [2]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

Creamos la funciona  de construir un prompt

In [3]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Creamos la funcion que llama al LLM

In [4]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

Juntamos las ultimas 3 en la funcion para hacer el RAG

In [5]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

Comprobamos que funciona

In [8]:
answer = rag('how do I implement LLM as a judge?')
print(answer)

To implement an LLM as a judge, follow these steps:

1. **Create an Evaluation Dataset**: This includes:
   - **Questions**: Inputs sent to the LLM.
   - **Target responses**: Approved accurate responses.
   - **New responses**: Responses generated by the system.
   - **Manual labels with explanations**: Indicate if the responses are correct or not.

2. **Install Required Libraries**:
   - Install Evidently using:
     ```python
     pip install evidently
     ```

3. **Import Required Modules**:
   ```python
   import pandas as pd
   import numpy as np
   from evidently import Dataset, DataDefinition, Report, BinaryClassification
   from evidently.llm.templates import BinaryClassificationPromptTemplate
   ```

4. **Set Up OpenAI API Key**:
   ```python
   import os
   os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
   ```

5. **Create the LLM Judge**: Design an LLM evaluator prompt tailored to your criteria for evaluation.

6. **Add Descriptors for Evaluation**: For instance, to evaluate co

Ahora vamos a crear una llamada al LLM estructurada.
Recibe un tipo de salida del moedlo Pydantic y usa responses.parse en vez de create y engancha el tipo de salida con el text_format de la llamada a openai

In [9]:
def llm_structured(
    user_prompt,
    output_type,
    instructions=None,
    model="gpt-4o-mini",
):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

Creamos el tipo de calendario de pydantic

In [10]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

Comprobamos que funciona

In [11]:

response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

Ahora vamos a hacer una respuesta estructurada para nuestro RAG.
- Creamos un modelo simple con dos campos

In [12]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool
    
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

Creamos nuestra funcion para RAG estructurado

In [13]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

La testeamos con una pregunta simple que si está en la documentacion

In [14]:
answer = rag_structured('how do I do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)

To perform LLM evaluations, you can use the following methods based on the context provided:

1. **W
True


Ahora probamos con una pregunta que no está en la documentación

In [15]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The documentation does not provide information on how to install Kafka on Windows. Please refer to K
False


Ahora vamos a hacer que la respuesta sea opcional

In [16]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool
RAGResponse.model_json_schema()



{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

Y volvemos a probar la respuesta que no existe.


In [17]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


Al ser opcional a veces generará answer y a veces no. Podríamos incluir algo en las instrucciones como:

In [18]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


En vez de poner las instrucciones para ese formateo en el prompt las podemos poner en el esquema

In [19]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

Podemos añadir descripciones incluso en los propios campos

In [21]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")


RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

Volvemos a testear la llamada, ahora lo que hemos metido en las propiedades y en el esquema le llega al LLM

In [22]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


Ahora vamos a añadir otros campos al modelo:
- confianza
- explicación de la confianza
- tipo de respuesta
- siguientes preguntas

In [24]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

Preguntamos por algo que no esta en la documentación

In [25]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)
answer

RAGResponse(answer='', found_answer=False, confidence=0.0, confidence_explanation='The context provided does not contain any information related to installing Kafka on Windows.', answer_type='reference', followup_questions=['What is Kafka?', 'Can you provide a tutorial on Kafka?', 'How do I configure Kafka after installation?'])

En vez de hacer cada campo opcional podemos hacer una estructura anidada. Cuando la respuesta ha sido encontrada damos toda la respuesta. Cuando no solo retornamos la explicación

In [27]:
from pydantic import model_validator

class AnswerNotFound(BaseModel):
    explanation: str

class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

Testeamos ahora con una pregunta que no está en la documentación

In [28]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided context does not include information on how to install Kafka on Windows.'), found_answer=False, answer=None)

Probamos ahora con una que si está en la documentación

In [29]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evals, follow these steps:\n\n1. **Installation and Import**: First, ensure you have the Evidently library installed. Use the following command:\n   ```bash\n   pip install evidently[llm] \n   ```\n   Then, import the required modules in Python:\n   ```python\n   import pandas as pd\n   from evidently.future.datasets import Dataset\n   from evidently.future.datasets import DataDefinition\n   from evidently.future.report import Report\n   from evidently.future.presets import TextEvals\n   from evidently.ui.workspace.cloud import CloudWorkspace\n   ```\n\n2. **Connect to Evidently Cloud**: Replace with your actual token:\n   ```python\n   ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")\n   ```\n\n3. **Create a Project**: Set up a project in Evidently Cloud:\n   ```python\n   project = ws.create_project("Your Project Name", org_id="YOUR_ORG_ID")\n   project.descrip

A veces el modelo va a generar los dos campos o ninguno, en este caso el validador del modelo fallará.
Hay que hacer el catch y reintentar

In [30]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
